In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

df = pd.read_csv("base.csv", sep=",", encoding="iso-8859-1")

def desvio_par(p1, p2):
    lat1, lon1 = p1
    lat2, lon2 = p2

    dist = np.sqrt((lat1 - lat2)**2 + (lon1 - lon2)**2)
    return dist / np.sqrt(2)

def corrigir_grupo(grupo):
    grupo = grupo.copy()
    n = len(grupo)

    print("\n==============================")
    print(f"Cidade: {grupo['cidade'].iloc[0]} - {grupo['estado'].iloc[0]}")
    print("==============================")

    if n < 3:
        grupo['corrigido'] = "Não"
        print("Poucos dados, não corrige")
        return grupo

    pontos = list(zip(grupo.index, grupo['latitude'], grupo['longitude']))
    pares = list(combinations(pontos, 2))

    resultados = []

    print("\n--- Desvios dos pares ---")

    for p1, p2 in pares:
        idx1, lat1, lon1 = p1
        idx2, lat2, lon2 = p2

        d = desvio_par((lat1, lon1), (lat2, lon2))

        print(f"Par ({idx1}, {idx2})  desvio = {d:.4f}")

        resultados.append({
            "idx1": idx1,
            "idx2": idx2,
            "desvio": d
        })

    df_pares = pd.DataFrame(resultados)

    limite_desvio = 10
    max_desvio = df_pares['desvio'].max()

    print("\nMáximo desvio no grupo:", round(max_desvio, 4))
    print("Limite definido:", limite_desvio)

    if max_desvio < limite_desvio:
        print("Não vai corrigir (sem outlier)")
        grupo['corrigido'] = "Não"
        return grupo

    print("Vai corrigir (outlier detectado)")

    limite = df_pares['desvio'].quantile(0.25)

    print("\nQuartil (limite pares bons):", round(limite, 4))

    pares_bons = df_pares[df_pares['desvio'] <= limite]

    print("\n--- Pares bons ---")
    print(pares_bons)

    if len(pares_bons) == 0:
        pares_bons = df_pares.nsmallest(max(1, n//2), 'desvio')

    freq = {}

    for _, row in pares_bons.iterrows():
        freq[row['idx1']] = freq.get(row['idx1'], 0) + 1
        freq[row['idx2']] = freq.get(row['idx2'], 0) + 1

    print("\n--- Frequência ---")
    print(freq)

    freq_series = pd.Series(freq)

    limiar = freq_series.mean()

    print("\nLimiar frequência:", round(limiar, 4))

    bons = list(set(pares_bons['idx1']).union(set(pares_bons['idx2'])))
    ruins = list(set(grupo.index) - set(bons))

    print("Pontos bons:", bons)
    print("Pontos ruins:", ruins)

    if len(bons) < 2:
        grupo['corrigido'] = "Não"
        print("Poucos bons, não corrige")
        return grupo

    lat_media = grupo.loc[bons, 'latitude'].mean()
    lon_media = grupo.loc[bons, 'longitude'].mean()

    print("\nMédia usada na correção:")
    print("Latitude:", lat_media)
    print("Longitude:", lon_media)

    grupo['corrigido'] = "Não"

    grupo.loc[ruins, 'latitude'] = lat_media
    grupo.loc[ruins, 'longitude'] = lon_media
    grupo.loc[ruins, 'corrigido'] = "Sim"

    return grupo


df_corrigido = (
    df.groupby(["cidade", "estado"], group_keys=False)
    .apply(corrigir_grupo)
)

print("\n================ FINAL ================\n")
print(df_corrigido[["cidade","latitude","longitude","corrigido"]])

df_corrigido.to_csv("base_corrigida.csv", index=False)